# Lekcija 18: Zavarovanje AI agentov s kriptografskimi potrdili

## Praktikum v zvezku

Ta zvezek vas vodi skozi štiri vaje:

1. **Podpišite svojo prvo potrdilo** za klic orodja agenta in ga preverite.
2. **Spremenite potrdilo** in opazujte neuspeh preverjanja.
3. **Ustvarite verigo treh potrdil** in potrdite integriteto verige.
4. **Ovojite klic orodja Microsoft Agent Framework**, tako da vsak ukrep ustvari potrdilo.

Vse kriptografske primitive so uvožene iz dobro vzdrževanih knjižnic (`pynacl` za Ed25519, `jcs` za RFC 8785 canonical JSON, `hashlib` iz standardne knjižnice Python za SHA-256). Logika potrdila je samotna Python koda, ki jo lahko berete in spreminjate.

Zaženite celice po vrsti. Vsak del je kratek in samostojen.


## Namestitev

Namestite obe odvisnosti. Obe imata permisivne licence (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Pomočna orodja

Ta dva pomočnika obdelujeta kodiranje base64url (brez zapolnitve) in SHA-256 zgoščevanje poljubnih objektov. Preostanek zapiska ohranjata osredotočen na samo logiko prejema.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Podpišite svoj prvi račun

Predstavljajte si, da je naš agent za **Contoso Travel** pravkar poiskal lete iz Sydneyja v Los Angeles za stranko. Želimo zabeležiti to orodje kot podpisan račun, da ga lahko bodoči pregledovalec preveri, ne da bi nam moral zaupati.

### Korak 1.1: Ustvarite ključ za podpisovanje

V proizvodnji bi bil agentov podpisni ključ shranjen v strojni varnostni modul (HSM), Azure Key Vault ali podobnem zaščitenem shrambi. Za to lekcijo ustvarimo nov ključ v pomnilniku.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Korak 1.2: Izdelava podatkovnega paketa potrdila

Podatkovni paket vsebuje vse, kar želimo, da potrdilo potrdi: kdo je ukrepal, kateri pripomoček, s katerimi argumenti, kaj je bilo vrnjeno, pod katero politiko in kdaj. Argumente in rezultat zgoščujemo, namesto da bi jih vključili neposredno v potrdilo, da potrdilo ne bi razkrilo občutljive vsebine.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Korak 1.3: Podpišite in sestavite prejem

Tri koraki:

1. Poenostavite vsebino z uporabo JCS, tako da dve implementaciji, ki ustvarjata isti logični prejem, ustvarita identične bajtne zapise.
2. Z zgoščevanjem (hashanjem) kanoničnih bajtov s SHA-256.
3. Podpišite zgoščevanje s privatnim ključem Ed25519.

Podpis se nato pripne na izvorno vsebino, da se ustvari končni prejem.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Korak 1.4: Preverite prejemek

Preverjanje obrne postopek. Odstranimo podpis, ponovno izračunamo kanonični zgošček in preverimo podpis glede na javni ključ v prejemku.

Revizor, ki izvaja to preverjanje, od nas ne potrebuje ničesar razen samega prejemka. Ni potrebe po klicu storitve, poizvedbi v imeniku ključev ali zaupanju.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Morali bi videti `Prejemek je veljaven: True`. Agent je ustvaril svoj prvi kriptografsko podpisan revizijski zapis.


## Sekcija 2: Sprememba računa

Glavni namen računov je, da so vidne spremembe na njih. Dokazali jih bomo.

Spremenili bomo natanko en znak na računu in opazovali, kako preverjanje spodleti.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Kaj se je pravkar zgodilo?

Ko smo spremenili `policy_id`, so se spremenili tudi kanonični bajti. SHA-256 zgoščena vrednost teh bajtov se je spremenila. Podpis (ki je bil narejen nad izvirno zgoščenko) se ne ujema več z novo zgoščenko. Preverjanje pravilno vrne `False`.

Ni mogoče spremeniti nobenega polja potrdila in hkrati doseči, da se potrdilo preveri, razen če napadalec nima zasebnega ključa. Dokler je zasebni ključ v ključnem trezorju, javni ključ pa je objavljen, je zamaskirati poseg nemogoče.

Poskusi sam: spremeni `tool_name` ali `agent_id` ali `timestamp` v zgornji celici in ponovno zaženi. Vsaka sprememba povzroči neveljavno potrdilo.


## Section 3: Povežite prejemke v verigo

En sam prejemek ščiti eno dejanje. Večina agentov izvede veliko dejanj. Da bi celoten zaporedje naredili vidno za ponarejanje, povežemo vsak prejemek s prejšnjim tako, da v novo obremenitev prejemka vključimo zgoščeno vrednost prejšnjega prejemka.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Če kdo odstrani ali preuredi prejemek, se veriga točno na tem mestu pretrga. Preverjanje kateregakoli kasnejšega prejemka spodleti, ker `previous_receipt_hash` ne ustreza več dejanski zgoščeni vrednosti njegovega predhodnika.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Zdaj pretrgajte verigo tako, da spremenite srednji račun in ponovno preverite. Spremenjeni račun ne uspe pri preverjanju podpisa, IN naslednji račun ne uspe pri preverjanju povezave verige (ker njegov `previous_receipt_hash` ne ustreza več zgoščeni vrednosti spremenjenega srednjega računa).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Prejemka 0 se še vedno preverja (ni bila spremenjena in nima predhodnika, na katerega bi se lahko zanašala). Prejemka 1 ne uspe preverjanja podpisa, ker smo spremenili `tool_args_hash`. Prejemka 2 ne uspe preverjanja povezave v verigi, ker je njen `previous_receipt_hash` izračunan glede na izvirno (zdaj spremenjeno) prejemko 1.

Tudi če napadalec ponovno podpiše spremenjeno prejemko 1 (kar ne more, če nima zasebnega ključa), bi nepravilnost povezave v verigi pri prejemki 2 še vedno razkrila manipulacijo. Da bi prikril spremembo, bi moral napadalec ponovno podpisati vsako prejemko od točke spremembe naprej, kar zahteva posedovanje zasebnega ključa.


## Oddelek 4: Ovejanje klica orodja agenta s podpisovanjem potrdila

V dejanski uporabi ne želite, da si vsak avtor agenta zapomni klic `make_receipt`. Želite, da je podpisovanje potrdila samodejno pri vsakem klicu orodja.

Tukaj je najpreprostejši vzorec: razred ovojnica, ki vzame katerokoli klicno funkcijo orodja in vrne njeno verzijo, ki izdaja potrdilo. To se prilagodi kateremukoli agentnemu okviru, vključno z Microsoft Agent Framework (`agent_framework.azure`).

Če nimate nameščenega projekta Azure AI Foundry, spodnji lokalni posnemovalec še vedno pokaže ta vzorec.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integracija z Microsoft Agent Framework

Zaviti `ReceiptedTool` zgoraj je neodvisen od ogrodja. Če ga želite uporabiti znotraj agenta, ki je zgrajen z Microsoft Agent Framework, registrirajte zavito funkcijo kot orodje. Osnutek (mock boste zamenjali z dejansko registracijo orodja Azure AI Foundry):

```python
# Psevdokoda, ki prikazuje obliko integracije.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Vi ste agent Contoso Travel ...",
#     tools=[receipted_lookup],   # orodje z ovojem, ne surova funkcija
# )
# response = agent.run("Poišči lete iz Sydneyja v Los Angeles junija.")
#
# # Po zagonu ima vsak klic orodja, ki ga je agent izvedel, podpisan prejemek:
# audit_chain = receipted_lookup.receipts
```

Agent framework ne potrebuje nobenih informacij o potrdilih. Podpisovanje potrdil je zavito okrog orodja, ne pa vgrajeno v ogrodje. Tako dodate izvor obstoječi kodi agenta, ne da bi prepisovali agenta.


## Povzetek in razširjeni izziv

Ste:

- Generirali par ključev Ed25519.
- Ustvarili in podpisali prejemek za klic orodja agenta.
- Preverili prejemek brez povezave z uporabo samo javnega ključa.
- Spremenili prejemek in opazovali neuspeh preverjanja.
- Ustvarili zaporedje treh prejemkov, vezanih s hešom.
- Spremenili srednji del verige in opazovali napako tako pri podpisu kot pri verigi.
- Ovili funkcijo orodja z avtomatskim podpisovanjem prejemkov.

**Razširjeni izziv.** Razširite shemo prejemka z poljem `request_id` (UUID za distribuirano sledenje). Posodobite `make_receipt`, da ga vključite, in potrdite, da se prejemki še vedno preverijo od začetka do konca. Nato spremenite to polje po podpisu in potrdite, da preverjanje ne uspe. To vas prisili, da razumete, kako vsak bajt kanoničnega kodiranja vpliva na podpis.

**Pomembna meja.** Prejemki dokazujejo tri stvari in samo tri stvari: avtorstvo (ta ključ je podpisal to vsebino), celovitost (vsebina se od podpisa ni spremenila) in vrstni red (ta prejemek je prišel po tistem prejemku). Ne dokazujejo, da je bila akcija agenta pravilna, da je bila politika, navedena v `policy_id`, dejansko ovrednotena ali da je agent sledil vsem pravilom. Prejemki so temelj. Upravljanje je sistem, ki ga zgradite na tem temelju.

Znova preberite README lekcije z upoštevanjem te meje. Najpogostejša napaka ekip pri prejemkih je predpostavka, da "imamo prejemke" pomeni "smo upravljani". To ne drži. Prejemki omogočajo revizijo vedenja agenta. Ne zagotavljajo njegove pravilnosti.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
